#Configuracao Inicial do databricks
Criacao do catalogo ˜mvp3_eng_dados˜

In [0]:
print("Catálogo atual:")
spark.sql("SELECT current_catalog()").show()

print("Schema atual:")
spark.sql("SELECT current_schema()").show()

print("Catálogos disponíveis:")
spark.sql("SHOW CATALOGS").show(truncate=False)

CATALOGO = "mvp3_eng_dados"

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOGO}
    COMMENT 'MVP-3 - Engenharia de Dados - PUC-Rio'
""")

print(f"✓ Catálogo {CATALOGO} criado ou já existente.")



Catálogo atual:
+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

Schema atual:
+----------------+
|current_schema()|
+----------------+
|         default|
+----------------+

Catálogos disponíveis:
+---------+
|catalog  |
+---------+
|samples  |
|system   |
|workspace|
+---------+

✓ Catálogo mvp3_eng_dados criado ou já existente.


In [0]:
SCHEMAS = [
    "dados_brutos",
    "bronze",
    "prata",
    "ouro"
]

for schema in SCHEMAS:
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS "
        f"{CATALOGO}.{schema}"
    )

    print(f"✓ {CATALOGO}.{schema}")

✓ mvp3_eng_dados.dados_brutos
✓ mvp3_eng_dados.bronze
✓ mvp3_eng_dados.prata
✓ mvp3_eng_dados.ouro


In [0]:
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOGO}.dados_brutos.volume_dados_brutos
    COMMENT 'Landing Zone dos arquivos originais do MVP-3'
""")

print("✓ Volume criado.")

✓ Volume criado.


In [0]:
spark.sql("SHOW SCHEMAS IN mvp3_eng_dados").show()

display(
    dbutils.fs.ls(
        "/Volumes/mvp3_eng_dados/"
        "dados_brutos/"
        "volume_dados_brutos"
    )
)

+------------------+
|      databaseName|
+------------------+
|            bronze|
|      dados_brutos|
|           default|
|information_schema|
|              ouro|
|             prata|
+------------------+



[]

In [0]:
# ============================================================
# VALIDAÇÃO DO AMBIENTE MVP-3
# ============================================================

CATALOGO = "mvp3_eng_dados"

SCHEMAS_ESPERADOS = {
    "dados_brutos",
    "bronze",
    "prata",
    "ouro"
}

VOLUME = "volume_dados_brutos"

print("=" * 60)
print("VALIDAÇÃO DO AMBIENTE MVP-3")
print("=" * 60)


# ------------------------------------------------------------
# 1. Validar catálogo
# ------------------------------------------------------------

catalogos = [
    row.catalog
    for row in spark.sql("SHOW CATALOGS").collect()
]

if CATALOGO in catalogos:
    print(f"✓ Catálogo encontrado: {CATALOGO}")
else:
    raise RuntimeError(
        f"ERRO: catálogo {CATALOGO} não encontrado."
    )


# ------------------------------------------------------------
# 2. Validar schemas
# ------------------------------------------------------------

schemas_existentes = {
    row.databaseName
    for row in spark.sql(
        f"SHOW SCHEMAS IN {CATALOGO}"
    ).collect()
}

for schema in SCHEMAS_ESPERADOS:

    if schema in schemas_existentes:
        print(f"✓ Schema encontrado: {schema}")
    else:
        raise RuntimeError(
            f"ERRO: schema {schema} não encontrado."
        )


# ------------------------------------------------------------
# 3. Validar Volume
# ------------------------------------------------------------

volumes = spark.sql(
    f"SHOW VOLUMES IN {CATALOGO}.dados_brutos"
).collect()

volumes_existentes = {
    row.volume_name
    for row in volumes
}

if VOLUME in volumes_existentes:
    print(f"✓ Volume encontrado: {VOLUME}")
else:
    raise RuntimeError(
        f"ERRO: volume {VOLUME} não encontrado."
    )


# ------------------------------------------------------------
# 4. Validar acesso ao Volume
# ------------------------------------------------------------

CAMINHO_VOLUME = (
    f"/Volumes/{CATALOGO}/"
    f"dados_brutos/{VOLUME}"
)

try:
    dbutils.fs.ls(CAMINHO_VOLUME)
    print("✓ Volume acessível pelo Databricks")
except Exception as erro:
    raise RuntimeError(
        f"ERRO ao acessar o Volume: {erro}"
    )


# ------------------------------------------------------------
# Resultado
# ------------------------------------------------------------

print()
print("=" * 60)
print("✓ AMBIENTE MVP-3 VALIDADO COM SUCESSO")
print("=" * 60)

print()
print(f"Catálogo : {CATALOGO}")
print("Schemas  : dados_brutos | bronze | prata | ouro")
print(f"Volume   : {VOLUME}")
print()
print("Próxima etapa: 01_ingestao_dados_brutos")

VALIDAÇÃO DO AMBIENTE MVP-3
✓ Catálogo encontrado: mvp3_eng_dados
✓ Schema encontrado: ouro
✓ Schema encontrado: bronze
✓ Schema encontrado: prata
✓ Schema encontrado: dados_brutos
✓ Volume encontrado: volume_dados_brutos
✓ Volume acessível pelo Databricks

✓ AMBIENTE MVP-3 VALIDADO COM SUCESSO

Catálogo : mvp3_eng_dados
Schemas  : dados_brutos | bronze | prata | ouro
Volume   : volume_dados_brutos

Próxima etapa: 01_ingestao_dados_brutos
